# am'

In [ ]:
///- --test static

open testing

In [ ]:
open rust
open rust_operators

## am'

### length

In [ ]:
inl length forall dim {int} el. (a : a dim el) : dim =
    a |> length

### index

In [ ]:
inl index forall dim {int} el. (i : dim) (a : a dim el) : el =
    backend_switch {
        Gleam = fun () => index a i
        Lua = fun () => index a i
        Fsharp = fun () => index a i
        Python = fun () => $'!a[!i]' : el
        Rust = fun () => index a i
        TypeScript = fun () => index a i
        Zig = fun () => index a i
        Lean = fun () => index a i
    }

### index_base

In [ ]:
inl index_base forall el. (i : int) (ar : array_base el) : el =
    ar |> a |> index i

### base

In [ ]:
inl base forall dim {int} el. ((a a) : a dim el) : array_base el =
    a

### base'

In [ ]:
inl base' forall el. ((a a) : a int el) : array_base el =
    a

### generic_equable

In [ ]:
inl generic_equable x1 x2 =
    if length x1 <>.. length x2
    then false
    else
        let rec 루프 i =
            if i < length x1
            then index i x1 = index i x2 && 루프 (i + 1)
            else true
        루프 0

### equable

In [ ]:
instance equable a dim {number; int} t = generic_equable

### append

In [ ]:
instance append a dim {int; number} t = am.append

In [1]:
///- --test

()

### map_base

In [ ]:
inl map_base forall t u. (fn : t -> u) (x : array_base t) : array_base u =
    a x
    |> am.map fn
    |> fun (a x : _ int _) => x

### collect

In [ ]:
inl collect forall t r. (fn : t -> a int r) (items : a int t) : a int r =
    items
    |> am.map fn
    |> am.fold (++) (a ;[])

### init_base

In [ ]:
inl init_base n fn : array_base _ =
    am.init n fn
    |> fun (a x : _ int _) => x

### init

In [ ]:
inl init n : array_base _ =
    init_base n id

### choose

In [ ]:
inl choose f l =
    (l, [])
    ||> am.foldBack fun x acc =>
        match f x with
        | Some y => y :: acc
        | None => acc
    |> listm.toArray

In [2]:
///- --test
///> fsharp
///> cuda
///> typescript
///> cuda

10
|> init
|> fun x => a x : _ int _
|> choose (fun x => if x % 2 = 0 then Some x else None)
|> _assert_eq (a' ;[ 0; 2; 4; 6; 8 ])

.fsx (fsharp)
__assert_eq { expected = [0, 2, 4, 6, 8] }
.py (cuda)
__assert_eq { expected = [0, 2, 4, 6, 8] }
.ts (typescript)
__assert_eq { expected = [0, 2, 4, 6, 8] }
.py (cuda)
__assert_eq { expected = [0, 2, 4, 6, 8] }

### sum

In [ ]:
inl sum a =
    a |> am.fold (+) 0

In [3]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

10
|> init
|> fun x => a x : _ int _
|> sum
|> _assert_eq 45

.fsx (fsharp)
__assert_eq { expected = 45 }
.py (cuda)
__assert_eq { expected = 45 }
.rs (rust)
__assert_eq { expected = 45 }
.ts (typescript)
__assert_eq { expected = 45 }

0

### init_series

In [ ]:
inl init_series start end inc : array_base _ =
    inl total = conv ((end - start) / inc) + 1
    am.init total (conv >> (*) inc >> (+) start)
    |> fun (a x : _ int _) => x

In [4]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

init_series 0i32 6 2
|> a
|> fun x => x : _ int _
|> _assert_eq (a ;[ 0i32; 2; 4; 6 ])

.fsx (fsharp)
__assert_eq { expected = [0, 2, 4, 6] }
.py (cuda)
__assert_eq { expected = [0, 2, 4, 6] }
.rs (rust)
__assert_eq { expected = [0, 2, 4, 6] }
.ts (typescript)
__assert_eq { expected = [0, 2, 4, 6] }

0

### head

In [ ]:
inl head (ar : a _ _) =
    if var_is ar || length ar > 0
    then ar |> index 0
    else error_type "The length of the array should be greater than 0."

### last

In [ ]:
inl last (ar : a _ _) =
    inl len = length ar
    if var_is ar || len > 0
    then ar |> index (len - 1)
    else error_type "The length of the array should be greater than 0."

In [5]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

10
|> init
|> fun x => a x : _ int _
|> last
|> _assert_eq 9

.fsx (fsharp)
__assert_eq { expected = 9 }
.py (cuda)
__assert_eq { expected = 9 }
.rs (rust)
__assert_eq { expected = 9 }
.ts (typescript)
__assert_eq { expected = 9 }

0

### try_pick

In [ ]:
inl try_pick forall t u. (fn : t -> option u) (array : a _ t) : option u =
    (array, None)
    ||> am.foldBack fun x acc =>
        match acc with
        | Some _ => acc
        | None => x |> fn

In [6]:
///- --test
///> fsharp
///> cuda
///> typescript
///> cuda

10
|> init
|> fun x => a x : _ int _
|> try_pick (fun x => if x = 5i32 then Some x else None)
|> _assert_eq (Some 5i32)

.fsx (fsharp)
__assert_eq { expected = Some(5) }
.py (cuda)
__assert_eq { expected = Some(5) }
.ts (typescript)
__assert_eq { expected = Some(5) }
.py (cuda)
__assert_eq { expected = Some(5) }

### indexed

In [ ]:
inl indexed forall t u {number}. (ar : array_base t) : array_base (u * t) =
    ((0, a ;[]), (a ar : _ int _))
    ||> am.fold fun (i, acc) x =>
        i + 1, acc ++ a ;[i, x]
    |> snd
    |> fun (a x : _ int _) => x

In [7]:
///- --test
///> fsharp
///> rust
///> typescript
///> cuda

am.init 3i32 ((*) 2)
|> fun (a x : _ int _) => x
|> indexed
|> _assert_eq' ;[0i32, 0; 1, 2; 2, 4]

.fsx (fsharp)
__assert_eq' { expected = [(0, 0), (1, 2), (2, 4)] }
.rs (rust)
__assert_eq' { expected = [(0, 0), (1, 2), (2, 4)] }
.ts (typescript)
__assert_eq' { expected = [(0, 0), (1, 2), (2, 4)] }
.py (cuda)
__assert_eq' { expected = [(0, 0), (1, 2), (2, 4)] }

0

### slice

In [ ]:
inl slice forall dim {int; number} el. from nearTo s : a dim el =
    am.slice { from nearTo } s

In [8]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

inl x : _ i32 _ = a ;[ 1i32; 2; 3 ]
x |> slice 0 0 |> _assert_eq (a ;[])
x |> slice 0 1 |> _assert_eq (a ;[ 1 ])
x |> slice 1 1 |> _assert_eq (a ;[])
x |> slice 1 2 |> _assert_eq (a ;[ 2 ])
x |> slice 2 2 |> _assert_eq (a ;[])
x |> slice 0 2 |> _assert_eq (a ;[ 1; 2 ])

.fsx (fsharp)
__assert_eq { expected = [] }
__assert_eq { expected = [1] }
__assert_eq { expected = [] }
__assert_eq { expected = [2] }
__assert_eq { expected = [] }
__assert_eq { expected = [1, 2] }
.py (cuda)
__assert_eq { expected = [] }
__assert_eq { expected = [1] }
__assert_eq { expected = [] }
__assert_eq { expected = [2] }
__assert_eq { expected = [] }
__assert_eq { expected = [1, 2] }
.rs (rust)
__assert_eq { expected = [] }
__assert_eq { expected = [1] }
__assert_eq { expected = [] }
__assert_eq { expected = [2] }
__assert_eq { expected = [] }
__assert_eq { expected = [1, 2] }
.ts (typescript)
__assert_eq { expected = [] }
__assert_eq { expected = [1] }
__assert_eq { expected = [] }
__assert_eq { expected = [2] }
__assert_eq { expected = [] }
__assert_eq { expected = [1, 2] }

0

### range

In [ ]:
union range dim =
    | Start : dim
    | End : (() -> dim) -> dim

inl range start end s =
    inl start, end =
        inl len () =
            s |> length |> conv
        match start, end with
        | Start start, End fn =>
            start, fn len
        | End start_fn, End end_fn =>
            start_fn len, end_fn len
    s |> slice (start |> convert.unbox) (end |> convert.unbox)

## rust

### vec

In [ ]:
nominal vec t =
    `(
        rust_type
            `($'Vec<`t>')
            `($'Vec<`t>')
            "type Vec<'T> = class end"
    )

### from_vec

In [ ]:
inl from_vec forall dim el. (vec : vec el) : a dim el =
    backend_switch {
        Rust = fun () =>
            inl vec = join vec
            $'Rc::new(RefCell::new(!vec))' : a dim el
    }

### from_vec_base

In [ ]:
inl from_vec_base forall el. (vec : vec el) : array_base el =
    backend_switch {
        Rust = fun () =>
            inl vec = join vec
            $'Rc::new(RefCell::new(!vec))' : array_base el
    }

### to_vec

In [ ]:
inl to_vec forall t. (ab : array_base t) : vec t =
    backend_switch {
        Rust = fun () => $'(!ab).borrow().clone()' : vec t
    }

### to_vec'

In [ ]:
inl to_vec' forall (t : * -> * -> *) u v. (x : t u v) : vec u =
    backend_switch {
        Rust = fun () => $'(!x).clone()' : vec u
    }

### to_vec''

In [ ]:
inl to_vec'' forall (t : * -> *) (u : * -> *) v. (x : t (u v)) : vec v =
    backend_switch {
        Rust = fun () => $'(!x).to_vec()' : vec v
    }

### to_vec'''

In [ ]:
inl to_vec''' forall t. (ab : array_base t) : vec t =
    !\\(ab, $'"to_vec($0)"')

### vec_push

In [ ]:
inl vec_push forall el. (el : el) (vec : vec el) : vec el =
    inl el = join el
    inl vec = join vec
    (!\($'"true; let mut !vec = !vec"') : bool) |> ignore
    (!\($'"true; !vec.push(!el)"') : bool) |> ignore
    !\($'"!vec"')

### vec_reverse

In [ ]:
inl vec_reverse forall el. (vec : vec el) : vec el =
    inl vec = join vec
    (!\($'"true; let mut !vec = !vec"') : bool) |> ignore
    (!\($'"true; !vec.reverse()"') : bool) |> ignore
    !\($'"!vec"')

### vec_retain

In [ ]:
inl vec_retain forall el. (fn : el -> bool) (vec : vec el) : vec el =
    inl vec = join vec
    inl fn = join fn
    (!\($'"true; let mut !vec = !vec"') : bool) |> ignore
    (!\($'"true; !vec.retain(|x| !fn(x.clone()))"') : bool) |> ignore
    !\($'"!vec"')

### vec_sort_by_key

In [ ]:
inl vec_sort_by_key forall el t. (fn : el -> t) (vec : vec el) : vec el =
    inl vec = join vec
    inl fn = rust.one_param_fn fn
    inl fn = join fn
    (!\($'"true; let mut !vec = !vec"') : bool) |> ignore
    (!\($'"true; !vec.sort_by_key(|x| !fn(x.clone()))"') : bool) |> ignore
    !\($'"!vec"')

### vec_extend

In [ ]:
inl vec_extend forall el. (el : vec el) (vec : vec el) : vec el =
    inl el = join el
    inl vec = join vec
    (!\($'"true; let mut !vec = !vec"') : bool) |> ignore
    (!\($'"true; !vec.extend(!el)"') : bool) |> ignore
    !\($'"!vec"')

### vec_mapi

In [ ]:
inl vec_mapi forall dim t u. (fn : dim -> t -> u) (ar : vec t) : vec u =
    inl fn i = rust.one_param_fn (fn i)
    inl fn = join fn
    inl ar = join ar
    !\($'"!ar.iter().enumerate().map(|(i, x)| !fn(i.try_into().unwrap())(x.clone())).collect::<Vec<_>>()"')

### vec_map

In [ ]:
inl vec_map forall t u. (fn : t -> u) (ar : vec t) : vec u =
    inl fn = rust.one_param_fn fn
    !\\((ar, fn), $'"$0.iter().map(|x| $1(x.clone())).collect::<Vec<_>>()"')

### vec_map'

In [ ]:
inl vec_map' forall t u. (fn : t -> u) (ar : vec t) : vec u =
    inl fn = rust.one_param_fn fn
    !\\((ar, fn), $'"$0.iter().map(|x| $1(x.clone())).collect::<Vec<_>>()"')

### vec_fold'

In [ ]:
inl vec_fold' forall t u. (fn : u -> t -> u) (init : u) (ar : vec t) : u =
    (!\\(ar, $'"true; let _vec_fold_ = $0.into_iter().fold(!init, |acc, x| { //"') : bool) |> ignore
    (!\\(fn !\($'"acc"') !\($'"x"'), $'"true; $0 })"') : bool) |> ignore
    !\($'"_vec_fold_"')

### vec_for_each

In [ ]:
inl vec_for_each forall t. (fn : t -> ()) (ar : vec t) : () =
    (!\\((ar, fn), $'"true; $0.iter().for_each(|x| { $1(x.clone()); }); //"') : bool) |> ignore

### vec_for_each'

In [ ]:
inl vec_for_each' forall t. (fn : t -> ()) (ar : vec t) : () =
    (!\\(ar, $'"true; $0.into_iter().for_each(|x| { //"') : bool) |> ignore
    (!\\(fn !\($'"x"'), $'$"true"') : bool) |> ignore
    (!\($'"true; }}); //"') : bool) |> ignore

### vec_for_each''

In [ ]:
inl vec_for_each'' forall t. (fn : t -> ()) (ar : vec t) : () =
    (!\\(ar, $'"true; $0.into_iter().for_each(|x| { //"') : bool) |> ignore
    (!\\(fn !\($'"x"'), $'$"true"') : bool) |> ignore
    (!\($'"true; }}); //"') : bool) |> ignore

### vec_for_each'''

In [ ]:
inl vec_for_each''' forall t. (fn : t -> ()) (ar : vec t) : () =
    (!\\(ar, $'"true; $0.into_iter().for_each(|x| { //"') : bool) |> ignore
    (!\\(fn !\($'"x"'), $'$"true"') : bool) |> ignore
    (!\($'"true; }); //"') : bool) |> ignore

### vec_filter

In [ ]:
inl vec_filter forall t. (fn : t -> bool) (ar : vec t) : vec t =
    inl fn = rust.one_param_fn fn
    !\\((ar, fn), $'"$0.iter().filter(|x| $1((*x).clone())).cloned().collect::<Vec<_>>()"')

### vec_len

In [ ]:
inl vec_len forall t. (vec : vec t) : convert.unativeint =
    backend_switch {
        Rust = fun () => $'((!vec).len() as usize)' : convert.unativeint
    }

### vec_chunks

In [ ]:
inl vec_chunks forall t. (n : i32) (vec : vec t) : vec (vec t) =
    !\\(vec, $'"$0.chunks(!n).map(|x| x.into_iter().map(|x| x.clone()).collect::<Vec<_>>()).collect::<Vec<_>>()"')

### slice

In [ ]:
nominal slice t =
    `(
        rust_type
            `($'Slice<`t>')
            `($'[`t]')
            "type Slice<'T> = class end"
    )

### slice'

In [ ]:
type slice'_switch el =
    {
        Gleam : $'Slice\'<`el>'
        Lua : $'Slice\'<`el>'
        Fsharp : $'Slice\'<`el>'
        Python : $'Slice\'<`el>'
        TypeScript : $'Slice\'<`el>'
        Cpp : $'Slice\'<`el>'
        CppHost : $'Slice\'<`el>'
        Delphi : $'Slice\'<`el>'
        Cuda : $'Slice\'<`el>'
        Rust : $'_'
    }
nominal slice' el dim =
    `(
        fsharp_global "type Slice'<'T> = class end"
        $'' : $'backend_switch `(slice'_switch el)'
    )

### slice''

In [ ]:
nominal slice'' el dim =
    `(
        rust_type
            `($'Slice\'_<`el>')
            `($'[`el; 10]')
            "type Slice'_<'T> = class end"
    )

### slice_singleton

In [ ]:
inl slice_singleton forall dim el. (x : option el) : slice' el dim =
    match x with
    | Some x => !\($'"[!x]"')
    | None =>
        !\($'"[\\\"\\\".to_string()]"') : slice' el dim

### slice_length

In [ ]:
inl slice_length forall t dim. (x : slice' t dim) : convert.unativeint =
    !\($'"!x.len()"')

### slice_range

In [ ]:
inl slice_range forall t dim.
    (start : range t) (end : range t) (s : slice' t dim) : rust.ref (slice' t dim)
    =
    inl len () =
        s |> slice_length
    inl start, (end : convert.unativeint) =
        match start, end with
        | Start start, End fn => start, (len >> convert.convert) |> fn |> convert.convert
        | End start_fn, End end_fn =>
            (len >> convert.convert) |> start_fn, (len >> convert.convert) |> end_fn |> convert.convert
    match start, end with
    | start, end when end |> convert.unbox =. len () => !\($'"&!s[!start..]"')
    | start, end => !\\((start, end), $'"&!s[$0..$1]"')

### new_slice

In [ ]:
inl new_slice forall el dim. (el : el) : slice' el dim =
    !\\(el, $'"[$0; @dim as usize]"')

### as_slice

In [ ]:
inl as_slice forall t. (x : array_base t) : rust.ref (slice t) =
    inl x = x |> to_vec
    !\($'"!x.as_slice()"')

### slice_to_vec

In [ ]:
inl slice_to_vec forall t. (slice : rust.ref (slice t)) : vec t =
    !\\(slice, $'"$0.iter().map(|x| *x).collect::<Vec<_>>()"')

### to_le_bytes

In [ ]:
inl to_le_bytes forall t. (x : t) : slice' u8 8 =
    backend_switch {
        Rust = fun () => $'(!x).to_le_bytes().to_vec()' : slice' u8 8
    }

### as_bytes

In [ ]:
inl as_bytes forall t. (x : t) : rust.ref (slice u8) =
    backend_switch {
        Rust = fun () => $'(!x).as_bytes()' : rust.ref (slice u8)
    }

### any

In [ ]:
inl any forall t. (fn : t -> bool) (source : array_base t) : bool =
    !\($'"!source.any(|x| !fn(x))"')

### iter_collect vec

In [ ]:
instance iter_collect vec = fun (iter : into_iterator u) =>
    !\\(iter, $'"$0.collect::<Vec<_>>()"')

instance iter_collect'' vec = fun (iter : into_iterator (t (u v))) =>
    !\\(iter, $'"$0.collect::<Vec<_>>()"')

### new_vec

In [ ]:
inl new_vec forall t. (items : list t) : vec t =
    backend_switch {
        Rust = fun () =>
            inl v : vec t = $'Vec::new()'
            items |> listm.iter fun x => $'!v.push(!x)' : ()
            v
    }

In [9]:
///- --test
///> rust

[ 0i32; 1 ]
|> new_vec
|> sm'.format_debug'
|> sm'.from_std_string
|> _assert_eq "[0, 1]"

__assert_eq { expected = [0, 1] }

0

## fsharp

### average

In [ ]:
inl average forall el {number}. (a : a _ el) : el =
    $'!a |> Array.average'

### distinct

In [ ]:
inl distinct forall dim el. (a : a dim el) : a dim el =
    if is_rust ()
    then
        inl a = $'!a ' : a int el
        inl r = $'Rc::new(RefCell::new(Vec::with_capacity(!a.borrow().len())))' : a int el
        loopw.for' {from=0; nearTo=length a} fun i =>
            inl x = index i a
            inl n = length r
            inl s = mut {j=0}
            loopw.while (fun () => s.j < n && index s.j r <> x) fun () =>
                s <- {j=s.j + 1}
            if s.j = n then $'!r.borrow_mut().push(!x)' : ()
        $'!r ' : a dim el
    else $'!a |> Array.distinct'

### skip

In [ ]:
inl skip forall dim el. (n : dim) (a : a dim el) : a dim el =
    run_target function
        | Rust => fun () => $'Rc::new(RefCell::new((!a).borrow().iter().skip((!n) as usize).cloned().collect::<Vec<_>>()))' : a dim el
        | Zig => fun () => $'(spiral_gpa.dupe(`el, (!a)[\@intCast(!n)..]) catch \@panic("out of memory"))' : a dim el
        | _ => fun () => $'!a |> Array.skip !n '

### skip_while

In [ ]:
inl skip_while forall dim el. (fn : el -> bool) (a : a dim el) : a dim el =
    if is_rust ()
    then $'Rc::new(RefCell::new((!a).borrow().iter().skip_while(|x| (!fn)((*x).clone())).cloned().collect::<Vec<_>>()))'
    else $'!a |> Array.skipWhile !fn '

### to_list_base'

In [ ]:
inl to_list_base' forall t. (items : array_base t) : listm'.list' t =
    backend_switch {
        Gleam = fun () => $'!items |> spiral_array_to_list' : listm'.list' t
        Lua = fun () => items |> convert.to : listm'.list' t
        Fsharp = fun () => $'!items |> Array.toList' : listm'.list' t
        Python = fun () => items |> convert.to : listm'.list' t
        Rust = fun () => $'Rc::new((!items).borrow().clone())' : listm'.list' t
        TypeScript = fun () => $'[...!items]' : listm'.list' t
        Zig = fun () => $'!items ' : listm'.list' t
    }

### to_list'

In [ ]:
inl to_list' forall dim {int} t. (items : a dim t) : listm'.list' t =
    items |> base |> to_list_base'

In [10]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

a' ;[ -3i32; 6 ]
|> to_list'
|> listm'.unbox
|> _assert_eq [ -3; 6 ]

.fsx (fsharp)
__assert_eq { expected = Cons(-3, Cons(6, Nil)) }
.py (cuda)
__assert_eq { expected = Cons(-3, Cons(6, Nil)) }
.rs (rust)
__assert_eq { expected = Cons(-3, Cons(6, Nil)) }
.ts (typescript)
__assert_eq { expected = Cons(-3, Cons(6, Nil)) }

0

### vec_collect

In [ ]:
inl vec_collect fn vec =
    ((;[] |> to_vec), ((vec |> from_vec : _ int _) |> to_list' |> listm'.unbox))
    ||> listm.fold fun acc x =>
        acc |> vec_extend (fn x)

### vec_collect_option

In [ ]:
inl vec_collect_option vec =
    ((;[] |> to_vec |> Ok), ((vec |> from_vec : _ int _) |> am.toList))
    ||> listm.fold fun acc x =>
        x
        |> resultm.unbox
        |> fun x =>
            match acc, x |> resultm.map optionm'.unbox with
            | Ok acc, Ok (Some x) => acc |> vec_extend x |> Ok
            | _, Error error => error |> Error
            | _ => acc

### vec_collect_into

In [ ]:
inl vec_collect_into forall (c : * -> * -> *) t e.
    (x : vec (c t e))
    : c (vec t) e
    =
    !\($'"!x.into_iter().collect()"')

### parallel_map

In [ ]:
inl parallel_map forall dim el el'. (fn : el -> el') (a : a dim el) : a dim el' =
    $'!a |> Array.Parallel.map !fn '

### map'

In [ ]:
inl map' forall dim el el'. (fn : el -> el') (a : a dim el) : a dim el' =
    $'!a |> Array.map !fn '

### sort_by

In [ ]:
inl sort_by forall dim el key. (fn : el -> key) (a : a dim el) : a dim el =
    inl fsharp () = $'!a |> Array.sortBy !fn ' : a dim el
    inl insertion_sort () =
        inl r = am.map id ($'!a ' : a int el)
        inl keys = am.map fn ($'!a ' : a int el)
        loopw.for' {from=1; nearTo=length r} fun i =>
            inl x = index i r
            inl k = index i keys
            inl s = mut {j=i}
            loopw.while (fun () => s.j > 0 && gt_is (comp (index (s.j - 1) keys) k)) fun () =>
                inl j = s.j
                set r j (index (j - 1) r)
                set keys j (index (j - 1) keys)
                s <- {j=j - 1}
            set r s.j x
            set keys s.j k
        $'!r ' : a dim el
    backend_switch {
        Gleam = fun () => fsharp ()
        Lua = fun () => fsharp ()
        Fsharp = fun () => fsharp ()
        Python = fun () =>
            inl keys = am.map fn ($'!a ' : a int el)
            $'!a[sorted(range(len(!a)), key=lambda __i: (lambda __k: (__k.tag, __k) if hasattr(__k, "tag") else __k)(!keys[__i]))]' : a dim el
        TypeScript = fun () =>
            inl keys = am.map fn ($'!a ' : a int el)
            global "function spiral_compare(a: any, b: any): number { if (typeof a === 'object' && a !== null && typeof b === 'object' && b !== null) { if (Array.isArray(a)) { for (let i = 0; i < Math.min(a.length, b.length); i++) { const c = spiral_compare(a[i], b[i]); if (c !== 0) return c; } return a.length - b.length; } if (a.tag !== b.tag) return a.tag - b.tag; for (const k of Object.keys(a)) { if (k === 'tag') continue; const c = spiral_compare(a[k], b[k]); if (c !== 0) return c; } return 0; } return a < b ? -1 : a > b ? 1 : 0; }"
            $'!a.map((_x: any, i: number) => i).sort((i: number, j: number) => spiral_compare(!keys[i], !keys[j])).map((i: number) => !a[i])' : a dim el
        Cpp = fun () => fsharp ()
        CppHost = fun () => fsharp ()
        Delphi = fun () => fsharp ()
        Cuda = fun () => fsharp ()
        Zig = insertion_sort
        Lean = insertion_sort
        Rust = fun () =>
            inl a = $'!a ' : a int el
            inl key_fn = join (rust.one_param_fn fn)
            inl r = $'Rc::new(RefCell::new(!a.borrow().clone()))' : a int el
            inl keys = $'Rc::new(RefCell::new(!a.borrow().iter().map(|x| !key_fn(x.clone())).collect::<Vec<_>>()))' : a int key
            loopw.for' {from=1; nearTo=length a} fun i =>
                inl x = index i r
                inl k = index i keys
                inl s = mut {j=i}
                loopw.while (fun () => s.j > 0 && gt_is (comp (index (s.j - 1) keys) k)) fun () =>
                    inl j = s.j
                    set r j (index (j - 1) r)
                    set keys j (index (j - 1) keys)
                    s <- {j=j - 1}
                set r s.j x
                set keys s.j k
            $'!r ' : a dim el
    }

### sort

In [ ]:
inl sort forall dim el. (a : a dim el) : a dim el =
    run_target function
        | Zig => fun () => sort_by id a
        | _ => fun () => $'!a |> Array.sort'

### sort_descending

In [ ]:
inl sort_descending forall dim el. (a : a dim el) : a dim el =
    $'!a |> Array.sortDescending'

### transpose

In [ ]:
inl transpose forall el. (a : array_base (array_base el)) : array_base (array_base el) =
    $'!a |> Array.transpose'

### try_item

In [11]:
inl try_item forall dim el. (i : i32) (a : a dim el) : option el =
    inl fsharp () : option el = $'!a |> Array.tryItem !i ' |> optionm'.unbox
    inl in_bounds_item () : option el =
        inl a = $'!a ' : a i32 el
        if i >= 0 && i < length a then Some (index i a) else None
    backend_switch {
        Gleam = fsharp
        Lua = fsharp
        Fsharp = fsharp
        Python = fsharp
        TypeScript = fsharp
        Cpp = fsharp
        CppHost = fsharp
        Delphi = fsharp
        Cuda = fsharp
        Zig = in_bounds_item
        Lean = in_bounds_item
        Rust = fun () =>
            inl x : optionm'.option' el = $'(if !i < 0 { None } else { !a.borrow().get(!i as usize).cloned() })'
            x |> optionm'.unbox
    }